# Resolve GBIF references

Match accepted species names for configured V3 organism groups that have at least one completed positive Phase 4 pathogen assessment against the GBIF Backbone, using the corresponding kingdom hint (Bacteria, Fungi, or Viruses). A link is recorded only for an exact canonical-name match at species rank; other GBIF matches remain diagnostics. Progress is checkpointed after every API batch.

By default, the notebook uses `run.organism_groups` from `configs/default.yaml` (currently `viruses`). It limits each selected group to species with at least one completed `candidate_pathogen_evidence=True` record in the Phase 4 pathogen decisions table. Change the group setting to `bacteria`, `fungi`, or any combination to resolve those pathogen candidates. Completed records are reused; set `FORCE_REFRESH = True` to repeat lookups for the selected groups.

In [1]:
from pathlib import Path
import os
import sys
import pandas as pd
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "pathogen_atlas_v3").is_dir():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise RuntimeError("Open this notebook from the Pathogen Atlas version3 repository.")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from pathogen_atlas_v3.common.io import load_yaml
from pathogen_atlas_v3.common.config import selected_organism_groups
CONFIG = load_yaml(PROJECT_ROOT / "configs/default.yaml")
ORGANISM_GROUPS = selected_organism_groups(CONFIG)
SPECIES_PATH = PROJECT_ROOT / CONFIG["paths"]["taxonomy_release"] / "species.parquet"
PATHOGEN_DECISIONS_PATH = PROJECT_ROOT / CONFIG["paths"]["release_root"] / "pathogen_decisions.parquet"
OUTPUT_PATH = PROJECT_ROOT / "data/06_references/release/species_gbif_links.csv"
FORCE_REFRESH = False

for required_path in (SPECIES_PATH, PATHOGEN_DECISIONS_PATH):
    if not required_path.exists():
        raise FileNotFoundError(f"Run Phases 1 and 4 first; required input is missing: {required_path}")

species = pd.read_parquet(SPECIES_PATH, columns=["organism_id", "organism_group", "accepted_name", "rank"])
decisions = pd.read_parquet(PATHOGEN_DECISIONS_PATH, columns=["organism_id", "organism_group", "candidate_pathogen_evidence", "status"])
pathogen_ids = set(decisions.loc[
    decisions.organism_group.isin(ORGANISM_GROUPS)
    & decisions.status.astype(str).eq("completed")
    & decisions.candidate_pathogen_evidence.fillna(False).astype(bool),
    "organism_id",
].astype(str))
scope = species.loc[
    species.organism_group.isin(ORGANISM_GROUPS)
    & species["rank"].astype(str).str.casefold().eq("species")
    & species.organism_id.astype(str).isin(pathogen_ids)
]
existing_count = 0
if OUTPUT_PATH.exists():
    previous = pd.read_csv(OUTPUT_PATH, keep_default_na=False)
    existing_count = int(previous.organism_group.isin(ORGANISM_GROUPS).sum()) if "organism_group" in previous else 0

display(Markdown(
    f"**Organism groups:** {', '.join(ORGANISM_GROUPS)}  ·  "
    f"**Species-rank records:** {scope.organism_id.nunique():,}  ·  "
    f"**Existing associations in scope:** {existing_count:,}  ·  "
    f"**Force refresh:** {FORCE_REFRESH}"
))


**Organism groups:** fungi  ·  **Species-rank records:** 440  ·  **Existing associations in scope:** 0  ·  **Force refresh:** False

## Resolve and save GBIF associations

The resolver writes a table keyed by `organism_id`, so identical accepted names in different domains remain separate. The same output file can be extended incrementally as you switch between bacteria, fungi, and viruses. Only the selected groups’ pathogen candidates are included.

In [2]:
from pathogen_atlas_v3.references.links import resolve_gbif_links

links = resolve_gbif_links(
    SPECIES_PATH,
    OUTPUT_PATH,
    ORGANISM_GROUPS,
    force_refresh=FORCE_REFRESH,
    organism_ids=pathogen_ids,
)
selected_links = links.loc[links.organism_group.isin(ORGANISM_GROUPS)]
found = selected_links.match_status.eq("found")
display(Markdown(
    f"**Saved:** `{OUTPUT_PATH.relative_to(PROJECT_ROOT)}`  ·  "
    f"**Records:** {len(selected_links):,}  ·  "
    f"**Exact GBIF species pages found:** {int(found.sum()):,}  ·  "
    f"**No exact page:** {int((~found).sum()):,}"
))
display(selected_links.head(12))


Resolving 440 records; reusing 257 completed rows.


Resolved 100/440 pending records.


Resolved 200/440 pending records.


Resolved 300/440 pending records.


Resolved 400/440 pending records.


Resolved 440/440 pending records.


Saved 697 associations to /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version3/data/06_references/release/species_gbif_links.csv


**Saved:** `data/06_references/release/species_gbif_links.csv`  ·  **Records:** 440  ·  **Exact GBIF species pages found:** 419  ·  **No exact page:** 21

,organism_id,organism_group,accepted_name,gbif_taxon_key,gbif_scientific_name,gbif_url,match_status,match_type,confidence
0,fungi:0007843d65dafa45c0d6,fungi,Nakaseomyces nivariensis,12260171,"Nakaseomyces nivariensis (Alcoba-Flórez, Ménde...",https://www.gbif.org/species/12260171,found,EXACT,99
1,fungi:006bae437be3781c7102,fungi,Neocucurbitaria unguis-hominis,9708559,Neocucurbitaria unguis-hominis (Punith. & M.P....,https://www.gbif.org/species/9708559,found,EXACT,99
2,fungi:009ec7bab8bb984f4bcb,fungi,Inonotus tropicalis,2520257,Inonotus tropicalis (M.J.Larsen & Lombard) T.W...,https://www.gbif.org/species/2520257,found,EXACT,99
3,fungi:011a0c22a6c1dc0cc8a9,fungi,Wickerhamiella pararugosa,10806031,"Wickerhamiella pararugosa (Nakase, Komag. & Fu...",https://www.gbif.org/species/10806031,found,EXACT,99
4,fungi:011b86f54ee97e2c0ab3,fungi,Naganishia albida,9721395,"Naganishia albida (Saito) Xin Zhan Liu, F.Y.Ba...",https://www.gbif.org/species/9721395,found,EXACT,99
5,fungi:018727e860669fa4fc72,fungi,Scedosporium boydii,7802843,"Scedosporium boydii (Shear) Gilgado, Gené, Can...",https://www.gbif.org/species/7802843,found,EXACT,99
6,fungi:0300cd9102a7e01b6272,fungi,Simplicillium sinense,,,,not_found,HIGHERRANK,94
7,fungi:0367452d738926eba164,fungi,Blastobotrys allociferrii,12249660,Blastobotrys allociferrii (Ueda-Nishim. & Mika...,https://www.gbif.org/species/12249660,found,EXACT,99
8,fungi:03701fa31aa5f45c7e99,fungi,Madurella mycetomatis,3455882,Madurella mycetomatis (Laveran) Brumpt,https://www.gbif.org/species/3455882,found,EXACT,99
9,fungi:04a19f50889fe2b5ef8b,fungi,Microsporum ferrugineum,2596056,Microsporum ferrugineum M.Ota,https://www.gbif.org/species/2596056,found,EXACT,98
